# Comparative SICR Modelling of Protest Demobilisation

**France 2023 (Pension Reform) vs Iran 2022 (Mahsa Amini)**

This notebook implements and calibrates the SICR (Susceptible–Informed–Committed–Retired) compartmental model
for protest dynamics, originally proposed by [Alsulami et al. (2022)](https://doi.org/10.3390/math10162957) and
extended with state-repression feedback by [Petrovskii et al. (2025)](https://doi.org/10.48550/arXiv.2501.01495).

We fit the model to two contrasting case studies using **differential evolution** and compare the resulting
parameter regimes to understand how repression intensity and social contagion shape protest trajectories.


## 1. Setup & Data Loading

We begin by importing the necessary libraries and loading the two datasets:

- **France 2023**: Weekly protest turnout estimates (midpoint of CGT and Ministry of Interior counts) during the pension reform strikes.
- **Iran 2022**: Daily cumulative arrest figures from the Mahsa Amini protests. Since direct crowd-size data is unavailable, we proxy active protesters via the daily *change* in arrests, scaled by a constant factor.


In [ ]:

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import differential_evolution
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')


In [ ]:

# Load Data
df_fr = pd.read_csv('data/france_2023_weekly.csv')
df_fr['date'] = pd.to_datetime(df_fr['date'])
t_fr = (df_fr['date'] - df_fr['date'].min()).dt.days.values
N_fr = df_fr['mid'].values

df_ir = pd.read_csv('data/iran_2022_weekly.csv')
df_ir['Date'] = pd.to_datetime(df_ir['Date'])
t_ir = (df_ir['Date'] - df_ir['Date'].min()).dt.days.values
daily_arrests = np.maximum(0, np.diff(df_ir['Number of Individuals Arrested'].values, prepend=0))
N_ir = daily_arrests * 50.0  # Proxy scaling


## 2. The SICR Model

The population is divided into four compartments:

| Compartment | Description |
|---|---|
| **S** (Susceptible) | Individuals who may join the protest |
| **I** (Informed) | Individuals aware of the protest but not yet fully committed |
| **C** (Committed) | Active, committed protesters |
| **R** (Retired) | Individuals who have left the movement |

The ODE system incorporates:
- **Social contagion** via contact rates \$ (I→S) and \$ (C→S)
- **Commitment deepening** at rate \$\chi\$
- **State repression** through a nonlinear crowd-size–dependent retirement rate \$\omega\$, modulated by a protest-day indicator function \(t)\$
- **Recovery** of retired individuals back into the susceptible pool at rate \$\gamma\$


In [ ]:

# Model and Simulation
def make_P_func(protest_days):
    def P(t):
        val = 0.0
        for pd in protest_days:
            val += 0.5 * (np.tanh(10 * (t - (pd - 0.5))) - np.tanh(10 * (t - (pd + 0.5))))
        return min(1.0, val)
    return P

def sicr_rhs(t, y, params, P_func):
    S, I, C, R = y
    b1, b2, chi, d11, d21, gamma, C0, n, eps12, eps22 = params
    Pt = P_func(t)
    omega = (d21 + eps22 * Pt) / (1.0 + ((I + C) / C0)**n)
    inflow = S * (b1 * I + b2 * C)
    dS = -inflow + gamma * R
    dI = inflow - chi * I - (d11 + eps12 * Pt) * I
    dC = chi * I - C * omega
    dR = (d11 + eps12 * Pt) * I + C * omega - gamma * R
    return [dS, dI, dC, dR]

def simulate_sicr(y0, params, t_span, protest_days, t_eval=None):
    P_func = make_P_func(protest_days)
    return solve_ivp(
        sicr_rhs, t_span, y0, args=(params, P_func),
        method='Radau', t_eval=t_eval, rtol=1e-3, atol=1e-6
    )


## 3. Parameter Calibration

We calibrate the model using **differential evolution** (a global optimiser) with a log-space mean squared error loss.
Two parameters are fixed to reduce identifiability issues on short time series:

| Parameter | Value | Rationale |
|---|---|---|
| \$\gamma\$ (recovery rate) | 0.01 | Slow return to susceptibility |
| \$ (nonlinearity exponent) | 4.0 | Strong crowd-size effect on repression |

Fitted parameters are cached to  to avoid re-running the expensive optimisation on every execution.


In [ ]:

# Fitting with Progress Bar
import json, os
os.makedirs("results", exist_ok=True)

PARAMS_FILE = "results/fitted_params.json"

def loss(params, t_obs, N_obs, S0, protest_days):
    full_params = list(params)
    full_params.insert(5, 0.010) # fixed gamma
    full_params.insert(7, 4.0)   # fixed n
    y0 = [S0, max(1, N_obs[0]*0.1), max(1, N_obs[0]*0.9), 0]
    
    sol = simulate_sicr(y0, full_params, [0, t_obs[-1]+10], protest_days=protest_days, t_eval=t_obs)
    if not sol.success or len(sol.t) != len(t_obs): return 1e9
    
    N_sim = np.maximum(1, sol.y[1] + sol.y[2])
    N_obs_safe = np.maximum(1, N_obs)
    return np.mean((np.log(N_sim) - np.log(N_obs_safe))**2)

def fit_model(name, t_obs, N_obs, S0, protest_days):
    bounds = [
        (1e-7, 1e-4), (1e-9, 1e-5), (0.01, 0.5), (0.01, 0.5), (0.05, 0.5),
        (10000, S0*0.5), (0.0, 1.0), (0.0, 1.0)
    ]
    
    pbar = tqdm(total=15, desc=f"Fitting {name}")
    def cb(xk, convergence=None):
        pbar.update(1)
        
    result = differential_evolution(
        loss, bounds, args=(t_obs, N_obs, S0, protest_days),
        seed=42, popsize=15, maxiter=15, polish=True, workers=1, callback=cb
    )
    pbar.close()
    
    full_params = list(result.x)
    full_params.insert(5, 0.010)
    full_params.insert(7, 4.0)
    return full_params

# Load cached params if available, otherwise fit
if os.path.exists(PARAMS_FILE):
    print("Loading cached parameters from", PARAMS_FILE)
    with open(PARAMS_FILE) as f:
        saved = json.load(f)
    params_fr = saved["france"]
    params_ir = saved["iran"]
else:
    print("Starting calibrations...")
    params_fr = fit_model("France", t_fr, N_fr, 4000000, t_fr)
    params_ir = fit_model("Iran", t_ir, N_ir, 6000000, t_ir)
    with open(PARAMS_FILE, "w") as f:
        json.dump({"france": params_fr, "iran": params_ir}, f, indent=2)
    print("Parameters saved to", PARAMS_FILE)


## 4. Results: Baseline Fit

The plots below compare the observed protest activity (circles) against the fitted SICR model (line) for both case studies.
The y-axis uses a logarithmic scale to capture the wide dynamic range of protest sizes.


In [ ]:

# Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

y0_fr = [4000000, max(1, N_fr[0]*0.1), max(1, N_fr[0]*0.9), 0]
sol_fr = simulate_sicr(y0_fr, params_fr, [0, t_fr[-1]], protest_days=t_fr)
ax1.plot(t_fr, np.maximum(1, N_fr), 'o', label='Observed (France)')
ax1.plot(sol_fr.t, np.maximum(1, sol_fr.y[1] + sol_fr.y[2]), label='Fitted')
ax1.set_yscale('log')
ax1.set_title('France 2023 Protests')
ax1.legend()

y0_ir = [6000000, max(1, N_ir[0]*0.1), max(1, N_ir[0]*0.9), 0]
sol_ir = simulate_sicr(y0_ir, params_ir, [0, t_ir[-1]], protest_days=t_ir)
ax2.plot(t_ir, np.maximum(1, N_ir), 'o', label='Observed (Iran)')
ax2.plot(sol_ir.t, np.maximum(1, sol_ir.y[1] + sol_ir.y[2]), label='Fitted')
ax2.set_yscale('log')
ax2.set_title('Iran 2022 Protests')
ax2.legend()

plt.show()


## 5. Discussion

The SICR model captures the broad demobilisation dynamics in both cases:

- **France 2023** shows a slower, more sustained pattern of protest with periodic mobilisation spikes aligned to national strike days.
- **Iran 2022** exhibits a sharp initial peak followed by rapid decay, consistent with intense state repression driving quick retirement from protest activity.

These contrasting trajectories highlight how the balance between social contagion (\, b_2\$) and repression intensity (\{21}, \epsilon_{22}\$) governs whether a protest movement sustains or collapses.
